## import ##

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from scipy import stats
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
import seaborn as sns
import matplotlib.colors as mcolors

In [ ]:
from sklearn.decomposition import PCA
from sklearn import manifold
from sklearn.manifold import TSNE
import umap.umap_ as umap
from sklearn import manifold
from sklearn import metrics

In [ ]:
from sklearn.impute import KNNImputer
import sys
from pathlib import Path
project_root = Path.cwd()
if not (project_root / "notebooks" / "main_fig.ipynb").is_file():
    raise RuntimeError("Run this notebook from the repository root.")
sys.path.append(str(project_root / "notebooks" / "utils"))  
from statutil.multivariate_test import t_integration as ting
from statutil.single_test import rankproduct as rp
import math

## directory ##

In [ ]:
data_root = str(project_root / "data")
table_root = str(project_root / "results" / "tables")
Path(table_root).mkdir(parents=True, exist_ok=True)
fig_root = str(project_root / "results" / "main_fig")
Path(fig_root).mkdir(parents=True, exist_ok=True)

## utils ##

In [ ]:
def save_panel(fig, filename, *, bbox_inches=None):
    """Save one computational panel as 900 dpi PNG and vector SVG."""
    target = Path(filename)
    target.parent.mkdir(parents=True, exist_ok=True)
    kwargs = {"bbox_inches": bbox_inches} if bbox_inches else {}
    fig.savefig(target, dpi=900, **kwargs)
    if target.suffix.lower() == ".png":
        fig.savefig(target.with_suffix(".svg"), format="svg", **kwargs)


def plot_line(df, name="ALT", figname=None):
    """Returns bar plot

    Parameters:
    ----------
    df : pd.DataFrame
        Information Data (ALT)

    """
    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111)

    for i in range(len(df)):
        x = [float(i.split("y")[1]) for i in df.columns.tolist()]
        y = np.array(df.loc[[i], :].values.tolist()[0])
        if i < 4:
            pass
            #plt.plot(x, y, color = 'gray', marker = 'o')
        elif i == 4:
            plt.plot(x, y, color = 'gray', marker = 'o', label="Control")
        elif i < 8:
            plt.plot(x, y, color = 'gray', marker = 'o')
        elif i < 14:
            #plt.plot(x, y, color = 'green', marker = 'v')
            pass
        elif i == 14:
            plt.plot(x, y, color = 'orangered', marker = 'v', label="TAA")
        elif i < 20:
            plt.plot(x, y, color = 'orangered', marker = 'v')
        elif i < 26:
            #plt.plot(x, y, color = 'deepskyblue', marker = '^')
            pass
        elif i == 26:
            plt.plot(x, y, color = 'mediumblue', marker = '^', label="MDA")
        elif i < 32:
            plt.plot(x, y, color = 'mediumblue', marker = '^')

    ax.set_xlabel("day", fontsize=17)
    ax.set_ylabel(name+" (U/L)", fontsize=17)
    ax.legend(loc="upper right", fontsize=15)
    ax.set_title("ALT changes over time", fontsize=21)

    #ax.set_ylim(0, 500)

    ax.grid(False)
    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)

    plt.tick_params(bottom=False, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname)
    plt.show()


def one_way_anova(df, day="day7", val_name="ALT", anova_thresh=0.05, tukey_thresh=0.05):
    data = {
    val_name: [df.loc[i, day] for i in df.index if i in [4, 5, 6, 7, 14, 15, 16, 17, 18, 19, 26, 27, 28, 29, 30, 31]], # sample indices for this comparison
    'group': ['Ctrl']*4 + ['TAA']*6 + ['MDA']*6
    }
    df_time = pd.DataFrame(data)
    df_time = df_time.dropna(subset=[val_name])
    group_A = [df_time.loc[i, val_name] for i in df_time.index if df_time.loc[i, "group"] == 'Ctrl']
    group_B = [df_time.loc[i, val_name] for i in df_time.index if df_time.loc[i, "group"] == 'TAA']
    group_C = [df_time.loc[i, val_name] for i in df_time.index if df_time.loc[i, "group"] == 'MDA']
    f_stat, p_value = f_oneway(group_A, group_B, group_C)

    print("F値:", f_stat)
    print("p値:", p_value)

    if p_value < anova_thresh:
        print("\n")
        tukey = pairwise_tukeyhsd(endog=df_time[val_name], groups=df_time['group'], alpha=tukey_thresh)
        print(tukey)


def calc_p(df, internal_standard="GAPDH"):
    df_sq = pd.DataFrame()
    # sampleとgeneのリストを取得
    sample_lst = [i for i in set(df.loc[:, "Sample"].tolist()) if str(i) != "nan"]
    gene_lst = [i for i in set(df.loc[:, "Target"].tolist()) if str(i) != "nan"]
    # SQのみを抽出
    for sample in sample_lst:
        for gene in gene_lst:
            df_sq.loc[gene, sample] = df[df["Sample"] == sample][df[df["Sample"] == sample]["Target"] == gene].loc[:, "Starting Quantity (SQ)"].iloc[-1]
    df_sq = df_sq.sort_index(axis=1)
    df_sq_is = (df_sq/df_sq.loc[internal_standard, :].tolist())
    df_ctrl_acta2 = df_sq_is.loc["Acta2", df_sq_is.columns.str.contains("Ctrl")]
    df_taa_acta2 = df_sq_is.loc["Acta2", df_sq_is.columns.str.contains("TAA")]
    df_mda_acta2 = df_sq_is.loc["Acta2", df_sq_is.columns.str.contains("MDA")]
    df_ctrl_col1a1 = df_sq_is.loc["Col1a1", df_sq_is.columns.str.contains("Ctrl")]
    df_taa_col1a1 = df_sq_is.loc["Col1a1", df_sq_is.columns.str.contains("TAA")]
    df_mda_col1a1 = df_sq_is.loc["Col1a1", df_sq_is.columns.str.contains("MDA")]
    t_taa_acta2, p_taa_acta2 =stats.ttest_ind(df_ctrl_acta2, df_taa_acta2, alternative="two-sided", equal_var=False)
    print("acta2 (ctrl, taa) :", p_taa_acta2)
    t_mda_acta2, p_mda_acta2 =stats.ttest_ind(df_ctrl_acta2, df_mda_acta2, alternative="two-sided", equal_var=False)
    print("acta2 (ctrl, mda) :",p_mda_acta2)
    t_taa_col1a1, p_taa_col1a1 =stats.ttest_ind(df_ctrl_col1a1, df_taa_col1a1, alternative="two-sided", equal_var=False)
    print("col1a1 (ctrl, taa) :",p_taa_col1a1)
    t_mda_col1a1, p_mda_col1a1 =stats.ttest_ind(df_ctrl_col1a1, df_mda_col1a1, alternative="two-sided", equal_var=False)
    print("col1a1 (ctrl, mda) :",p_mda_col1a1)
    return df_sq


def sq_plot(df_lst, internal_standard="", target_gene="", y_max=0, figname=None):
    fig = plt.figure(figsize=(7, 7))
    for i, df_sq in enumerate(df_lst):
        df_sq_is = (df_sq/df_sq.loc[internal_standard, :].tolist()).T
        
        df_ctrl = df_sq_is.loc[[i.split("_")[0] == "Ctrl" for i in df_sq_is.index.tolist()], :]
        df_ctrl = df_ctrl.loc[:, [target_gene]]

        df_taa = df_sq_is.loc[[i.split("_")[0] == "TAA" for i in df_sq_is.index.tolist()], :]
        df_taa = df_taa.loc[:, [target_gene]]

        df_mda = df_sq_is.loc[[i.split("_")[0] == "MDA" for i in df_sq_is.index.tolist()], :]
        df_mda = df_mda.loc[:, [target_gene]]

        ax = fig.add_subplot(1, 2, i+1)
        ax2 = ax.twinx()

        width = 0.25
        x_position = np.arange(len(df_ctrl.columns))
        error_bar_set = dict(lw = 1, capthick = 1, capsize = 4)
        ax.bar(x_position-width, df_ctrl.mean(), yerr=df_ctrl.std(), tick_label=df_ctrl.columns, error_kw=error_bar_set, color='gray', width=0.97*width, label="Ctrl", alpha=.8)
        ax.bar(x_position+width, df_mda.mean(), yerr=df_mda.std(), tick_label=df_ctrl.columns, error_kw=error_bar_set, color='royalblue', width=0.97*width, label="MDA", alpha=.8)
        ax.bar(x_position, df_taa.mean(), yerr=df_taa.std(), tick_label=df_ctrl.columns, error_kw=error_bar_set, color='coral', width=0.97*width, label="TAA", alpha=.8)

        ax.legend(fontsize=18, loc="upper right")
        plt.grid(False)
        [ax.spines[side].set_visible(False) for side in ['right','top']]

        # jitter plot
        df_melt1 = pd.melt(df_ctrl)
        df_melt1['dose'] = 'Ctrl'
        df_melt2 = pd.melt(df_taa)
        df_melt2['dose'] = 'TAA'
        df_melt3 = pd.melt(df_mda)
        df_melt3['dose'] = 'MDA'
        df_melt = pd.concat([df_melt1, df_melt2, df_melt3], axis=0)

        sns.stripplot(x='variable', y='value', data=df_melt, hue='dose', dodge=True, jitter=True, s=7, linewidth=1, edgecolor="black", palette=['black', 'orangered', 'mediumblue'], ax=ax2)

        if i == 0:
            ax.set_ylabel("SQ (ratio, IS = "+internal_standard+")", fontsize=20)
            ax.set_xticklabels(["day14"])
            ax.get_legend().remove()         
        elif i == 1:
            ax.set_xticklabels(["day28"])

        ax.tick_params(axis='x', labelsize=16, direction='in', width=0)
        ax.tick_params(axis='y', labelsize=16)

        ax.set_ylim(0,y_max)
        ax2.set_ylim(0,y_max)

        ax2.axis("off")
        ax2.get_legend().remove()

    fig.suptitle(target_gene, fontsize=24)
    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname, bbox_inches="tight")
    plt.show()


def value_plot(df_total, target="ALT", comp_lst=[], c_lst=[], title=None, lim=None, figname=None):
    # dfの準備 (processing)
    df_target_lst = [df_total.loc[df_total['group'] == comp, ['group', 'day', target]] for comp in comp_lst] #各化合物ごとのdfのlstを作成
    df_target_all = pd.concat(df_target_lst) #つなげる
    n_comp = len(comp_lst)

    # Figの基本設定
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(1, 1, 1)
    ax2 = ax.twinx() # axと同じ領域にax2を重ねる
    width = 1/(n_comp+1) # compが増えても対応できるようにこの値
    x_position = np.arange(len(set(df_target_all.loc[:, "day"]))) # xの位置
    error_bar_set = dict(lw = 1, capthick = 1, capsize = 4) #エラーバーの設定

    # barプロット (compの数に応じて変更できるようにfor文)
    for n, df_target in enumerate(df_target_lst):
        target_value_mean = [df_target.loc[df_target["day"] == day, [target]].mean()[target] for day in sorted(list(set(df_target.loc[:, "day"])))] # あるcompでのtarget (血液生化学値の項目)の平均
        target_value_std = [df_target.loc[df_target["day"] == day, [target]].std()[target] for day in sorted(list(set(df_target.loc[:, "day"])))] # あるcompでのtargetのstd
        loc = (1-n_comp)/2 + n
        ax.bar(x_position + width*loc, target_value_mean, 
            yerr=target_value_std, 
            tick_label=["day"+str(day) for day in set(df_target.loc[:, "day"])], 
            error_kw=error_bar_set, 
            color=c_lst[n], 
            width=0.97*width, label=comp_lst[n], alpha=.8) #平均をプロット
        
    ax.legend(fontsize=18, loc="upper left") # legendの大きさと位置
    plt.grid(False) # 外枠を削除
    [ax.spines[side].set_visible(False) for side in ['right','top']] # 外枠を右と下に追加

    # stripプロット
    named_colors = mcolors.get_named_colors_mapping()
    point_c_lst = [tuple([max(i-0.2, 0.0) for i in mcolors.to_rgb(named_colors[c])]) for c in c_lst] #各点の色をc_lstから生成 (少し濃くする)
    sns.stripplot(x='day', y=target, data=df_target_all, hue='group', dodge=True, jitter=True, s=7, linewidth=1, edgecolor="black", palette=point_c_lst, ax=ax2) # 各個体の値をプロット

    # control line
    if target == "GGT":
        l = len(set(df_target.loc[:, "day"]))
        n = len(comp_lst)
        ax.hlines(y=10, xmin=-n*width/2-(0.5*width), xmax=(l*width*n)+width*(l+1)/2, color="black", linestyles='dashed', linewidth=1.5, alpha=.8, zorder=1)

    # ax (or ax2)の細かな設定
    ax.set_xticklabels(["day"+str(day) for day in set(df_target.loc[:, "day"])], rotation=45, fontsize=20) # xのラベル設定
    ax.set_ylabel(title, fontsize=22)
    ax.set_title(target, fontsize=24)
    if lim is None:
        lim = 1.1*df_target_all.loc[:, target].max()
    if target == "GGT":
        ax.set_xlim(-n*width/2-(0.5*width),(l*width*n)+width*(l+1)/2) # axとax2でlimを揃えないとずれる
        ax2.set_xlim(-n*width/2-(0.5*width),(l*width*n)+width*(l+1)/2)
    ax.set_ylim(0,lim) # axとax2でlimを揃えないとずれる
    ax2.set_ylim(0,lim)
    ax2.axis("off") 
    ax2.get_legend().remove() # ax2 (点)の方のlegendを消す
    [ax2.spines[side].set_visible(False) for side in ['right','top']]
    ax.grid(False) # hide the plot grid
    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')
    ax.tick_params(axis='y', labelsize=18)

    # plt側の細かな設定
    plt.tick_params(bottom=False, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    if figname:
        save_panel(plt.gcf(), figname, bbox_inches="tight")

    plt.show()


"""
Significance annotation for the immune-cell violin plots (Supplementary Fig. 4).

Two pieces:

1. compute_immune_stats(...)  — runs the whole test family once and returns a
   lookup table. It must see every population and every day, because the
   Benjamini-Hochberg correction is applied across all 55 Kruskal-Wallis tests
   (11 populations x 5 time points), not within a single panel.

2. violinplot_twogroup(...)   — draws the figure and adds significance annotations
   from the lookup table.

Usage
-----
    stats_all = compute_immune_stats(df_all, populations=POPS)   # df_all includes Control
    violinplot_twogroup(df_dili, immune="CD8T", celltype="lymph",
                        stats_result=stats_all, figname=...)

Dependency: scikit-posthocs (pinned in environment/requirements-lock.txt)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sph


# --------------------------------------------------------------------------
# statistics
# --------------------------------------------------------------------------

def compute_immune_stats(df_all, populations, group_col="group", day_col="day",
                         groups=("Control", "MDA", "TAA"), alpha=0.05):
    """
    Run the full test family over every population x time point.

    Returns
    -------
    dict: stats[population][day] = {"kw_p", "kw_q", "significant", "pairs"}
          where "pairs" maps frozenset({g1, g2}) -> Dunn p-value, and is empty
          when the BH-corrected Kruskal-Wallis p-value was not below alpha.
    """
    days = sorted(df_all[day_col].unique())

    # --- omnibus test -----------------------------------------------------
    # Kruskal-Wallis: the rank-based counterpart of the one-way ANOVA used for
    # the plasma biochemical data. Chosen because these values are proportions,
    # so the variance scales with the mean and differs by more than two orders
    # of magnitude between populations, and because the small, unequal group
    # sizes (n = 3-8) do not permit verification of normality or homogeneity of
    # variance.
    rows = []
    for pop in populations:
        for day in days:
            sub = df_all[df_all[day_col] == day]
            samples = [sub.loc[sub[group_col] == g, pop].dropna().values
                       for g in groups]
            H, p = sp_stats.kruskal(*samples)
            rows.append({"pop": pop, "day": day, "H": H, "p": p})
    kw = pd.DataFrame(rows)

    # --- correction across the whole family --------------------------------
    # Benjamini-Hochberg over all 11 populations x 5 time points (55 tests),
    # not within a panel, because the populations were screened together.
    kw["q"] = multipletests(kw["p"].values, method="fdr_bh")[1]

    # --- post hoc ----------------------------------------------------------
    # Dunn's test, the rank-based counterpart of Tukey's HSD, with Holm
    # correction over the three pairwise comparisons within each test. Run only
    # where the corrected omnibus p-value was below alpha, so that the post hoc
    # comparisons are conditioned on a significant Kruskal-Wallis result.
    out = {pop: {} for pop in populations}
    for _, r in kw.iterrows():
        entry = {"kw_p": r["p"], "kw_q": r["q"],
                 "significant": bool(r["q"] < alpha), "pairs": {}}
        if entry["significant"]:
            sub = df_all[df_all[day_col] == r["day"]][[group_col, r["pop"]]].dropna()
            sub.columns = ["group", "value"]
            m = sph.posthoc_dunn(sub, val_col="value", group_col="group",
                                 p_adjust="holm")
            for i, g1 in enumerate(groups):
                for g2 in groups[i + 1:]:
                    entry["pairs"][frozenset((g1, g2))] = float(m.loc[g1, g2])
        out[r["pop"]][r["day"]] = entry
    return out


def _stars(p, ns_label=""):
    """p-value to symbol, matching the convention used elsewhere in the paper."""
    if p is None or np.isnan(p):
        return ns_label
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ns_label


# --------------------------------------------------------------------------
# plot
# --------------------------------------------------------------------------

def _symbols(p, symbol="*", ns_label=""):
    """p-value to a repeated symbol, so that vs-control and between-group
    comparisons can be distinguished by glyph as well as by position."""
    if p is None or np.isnan(p):
        return ns_label
    if p < 0.001:
        return symbol * 3
    if p < 0.01:
        return symbol * 2
    if p < 0.05:
        return symbol
    return ns_label


def violinplot_twogroup_v2(df_dili, immune='', celltype='', figname=None,
                           stats_result=None, control_label="Control",
                           ns_label="", annot_fontsize=14,
                           control_symbol="*", between_symbol="#"):
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(1, 1, 1)

    flatui = ["coral", "royalblue"]
    sns.set_palette(flatui)
    sns.violinplot(data=df_dili, x='day', y=immune, inner=None, hue='group', split=True)

    if celltype == "lymph":
        fig.legend(frameon=True, loc=(0.755, 0.75), fontsize=10, framealpha=1)
    elif celltype == "mye":
        fig.legend(frameon=True, loc=(0.25, 0.75), fontsize=10, framealpha=1)

    flatui = ["orangered", "mediumblue"]
    sns.set_palette(flatui)
    # Draw every non-missing animal at deterministic, distinct x positions.
    day_levels = list(pd.unique(df_dili['day']))
    hue_levels = list(pd.unique(df_dili['group']))
    point_colors = dict(zip(hue_levels, flatui))
    for xi, day in enumerate(day_levels):
        for gi, group in enumerate(hue_levels):
            vals = df_dili.loc[(df_dili['day'] == day) &
                               (df_dili['group'] == group), immune].dropna().to_numpy()
            offsets = np.linspace(-0.06, 0.06, len(vals)) if len(vals) > 1 else np.zeros(len(vals))
            center = xi + (-0.13 if gi == 0 else 0.13)
            ax.scatter(center + offsets, vals, s=25, c=point_colors[group],
                       edgecolors="black", linewidths=1, zorder=3)

    ax.get_legend().remove()

    plt.grid(False)
    plt.tick_params(bottom=True, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')
    ax.set_title(immune, fontsize=24)
    ax.set_xlabel("day", fontsize=20)
    ax.set_ylabel("population", fontsize=20)
    ax.tick_params(axis='x', labelsize=16)
    ax.tick_params(axis='y', labelsize=16)

    # ---- significance annotation ----------------------------------------
    # Two kinds of comparison are drawn, distinguished both by glyph and by
    # vertical position:
    #   control_symbol (#)  each plotted group vs the concurrent control,
    #                       placed over its own half of the split violin
    #   between_symbol (*)  MDA vs TAA, drawn as a bracket above them
    if stats_result is not None and immune in stats_result:
        days_plotted = [t.get_text() for t in ax.get_xticklabels()]
        hue_levels = list(pd.unique(df_dili['group']))          # split order: left, right
        ymin, ymax = ax.get_ylim()
        span = ymax - ymin
        top = ymax                                              # tracks the highest annotation

        for xi, day_label in enumerate(days_plotted):
            # x tick labels are strings; match them back to the keys used in stats
            day_key = next((d for d in stats_result[immune]
                            if str(d) == day_label), None)
            if day_key is None:
                continue
            entry = stats_result[immune][day_key]
            if not entry["significant"]:
                continue

            # height reference: top of the observed values on this day
            vals = df_dili.loc[df_dili['day'].astype(str) == day_label, immune].dropna()
            base = float(vals.max()) if len(vals) else ymin
            step = span * 0.09          # annotation spacing

            # --- vs concurrent control, one per half -------------------------
            for k, g in enumerate(hue_levels):
                p = entry["pairs"].get(frozenset((control_label, g)))
                lab = _symbols(p, control_symbol, ns_label)
                if lab:
                    y = base + step * 0.35
                    ax.text(xi + (-0.20 if k == 0 else 0.20), y, lab,
                            ha="center", va="bottom",
                            fontsize=annot_fontsize, clip_on=False)
                    top = max(top, y + step * 0.5)

            # --- MDA vs TAA, as a bracket well clear of the symbols ----------
            if len(hue_levels) == 2:
                p = entry["pairs"].get(frozenset(tuple(hue_levels)))
                lab = _symbols(p, between_symbol, ns_label)
                if lab:
                    y = base + step * 1.6
                    h = span * 0.018
                    ax.plot([xi - 0.20, xi - 0.20, xi + 0.20, xi + 0.20],
                            [y, y + h, y + h, y], lw=1.0, c="black",
                            clip_on=False)
                    ax.text(xi, y + h * 1.1, lab, ha="center", va="bottom",
                            fontsize=annot_fontsize, clip_on=False)
                    top = max(top, y + h + step * 0.6)

        ax.set_ylim(ymin, max(ymax, top))
    # ---------------------------------------------------------------------

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname, bbox_inches="tight")
    plt.show()

def stats_to_df(stats_result):
    rows = []
    for pop, per_day in stats_result.items():
        for day, entry in per_day.items():
            base = {
                "population": pop,
                "day": day,
                "kw_p": entry["kw_p"],
                "kw_q_BH": entry["kw_q"],
                "kw_significant": entry["significant"],
            }
            if entry["pairs"]:
                for pair, p in entry["pairs"].items():
                    g1, g2 = sorted(pair)
                    rows.append({**base,
                                 "comparison": f"{g1} vs {g2}",
                                 "dunn_p_holm": p,
                                 "dunn_significant": p < 0.05})
            else:
                rows.append({**base, "comparison": None,
                             "dunn_p_holm": None, "dunn_significant": None})
    df = pd.DataFrame(rows)
    return df.sort_values(["population", "day", "comparison"]).reset_index(drop=True)


# 第一主成分と第二主成分でプロットする
def PCAplot(feature, bacth_lst_ex, batch_c_ex, xnum=0, ynum=1, title="RNAseq batch check", figname=None):
    fig, ax = plt.subplots(figsize=(10, 7))
    labelset = []
    ap = labelset.append
    for i in range(len(bacth_lst_ex)):
        if bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]) not in labelset:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            ax.scatter(feature[:, xnum][i], feature[:, ynum][i], alpha=1, color=color_map((batch_c_ex[i][1])/2+0.4), label=bacth_lst_ex[i] + " day" + str(int(10*batch_c_ex[i][1])))
            ap(bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]))
        else:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            ax.scatter(feature[:, xnum][i], feature[:, ynum][i], alpha=1, color=color_map((batch_c_ex[i][1])/2+0.4), label="")
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    ax.set_title(title, fontsize=18)
    ax.set_xlabel("PC"+str(xnum+1), fontsize=15)
    ax.set_ylabel("PC"+str(ynum+1), fontsize=15)

    #ax.set_xlim(-130,410)

    ax.grid(False)
    [ax.spines[side].set_visible(False) for side in ['right','top']]

    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')

    plt.tick_params(bottom=True, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname, bbox_inches="tight")
    plt.show()


# 第一主成分と第二主成分でプロットする
def tSNEplot(feature, bacth_lst_ex, batch_c_ex, xnum=0, ynum=1, title="RNAseq batch check", figname=None):
    fig, ax = plt.subplots(figsize=(10, 7))
    labelset = []
    ap = labelset.append
    for i in range(len(bacth_lst_ex)):
        if bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]) not in labelset:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, color=color_map((batch_c_ex[i][1])/2+0.4), label=bacth_lst_ex[i] + " day" + str(int(10*batch_c_ex[i][1])))
            ap(bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]))
        else:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, color=color_map((batch_c_ex[i][1])/2+0.4), label="")
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    ax.set_title(title, fontsize=18)
    ax.set_xlabel("axis"+str(xnum+1), fontsize=15)
    ax.set_ylabel("axis"+str(ynum+1), fontsize=15)

    ax.grid(False)
    [ax.spines[side].set_visible(False) for side in ['right','top']]

    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')

    plt.tick_params(bottom=True, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname, bbox_inches="tight")
    plt.show()


# 第一主成分と第二主成分でプロットする
def UMAPplot(feature, bacth_lst_ex, batch_c_ex, xnum=0, ynum=1, title="RNAseq batch check", figname=None):
    fig, ax = plt.subplots(figsize=(10, 7))
    labelset = []
    ap = labelset.append
    for i in range(len(bacth_lst_ex)):
        if bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]) not in labelset:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, color=color_map((batch_c_ex[i][1])/2+0.4), label=bacth_lst_ex[i] + " day" + str(int(10*batch_c_ex[i][1])))
            ap(bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]))
        else:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, color=color_map((batch_c_ex[i][1])/2+0.4), label="")
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    ax.set_title(title, fontsize=18)
    ax.set_xlabel("axis"+str(xnum+1), fontsize=15)
    ax.set_ylabel("axis"+str(ynum+1), fontsize=15)

    ax.grid(False)
    [ax.spines[side].set_visible(False) for side in ['right','top']]

    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')

    plt.tick_params(bottom=True, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname, bbox_inches="tight")
    plt.show()


def meta_viz(visualizations, projection_method=None):
    '''
    Inputs:
       visualizations [list of numpy arrays]
        - list of numpy arrays of size n x 2 (columns being the two-dimensional coordinates of visualization)
        - each row should be matched to the same observation/sample across all elements in the list
       projection_method [str or None]
        - options are "tSNE" or "MDS" for reprojecting the meta-distance matrix into a meta-visualization

    Approach:
        1. Computes euclidean distance matrices from 2D embeddings
        2. Ensembles the euclidean distance according to eigenvector with largest eigenvalue
        3. Uses projection_method ["MDS", "tSNE"] to transform euclidean distance matrix to 2D embeddings

    Returns:
        meta_visualization [numpy array] - size nx2 two-dimensional visualization genereated using the meta-visualization approach
            - returned only if projection_method is not None
        meta_distances [numpy array] - size nxn pairwise euclidean distance matrix generated using meta-visualization approach
    '''
    # define number of samples
    n = visualizations[0].shape[0]
    K = len(visualizations)

    # Iterate and record distance matrices
    X_distance_matrix_list = []

    # compute pairwise distance matrix
    for X_embedded in visualizations:
        assert X_embedded.shape[0] == n, "All visualization arrays need to have the same number of rows"
        assert X_embedded.shape[1] == 2, "All visualization arrays need to have two columns"
        X_distance = metrics.pairwise_distances(X_embedded)
        X_distance_matrix_list.append(X_distance)

    # Compute weights for meta-visualization
    meta_distances = np.zeros((n,n))
    weights = np.zeros((n,K))
    for j in range(n):
        # fill in comparison matrix
        comp_mat = np.zeros((K,K))
        for i in range(K):
            for k in range(K):
                comp_mat[i,k] = np.sum(X_distance_matrix_list[k][:,j]*X_distance_matrix_list[i][:,j])/np.sqrt(np.sum(X_distance_matrix_list[k][:,j]**2))/np.sqrt(np.sum(X_distance_matrix_list[i][:,j]**2))
        # Eigenscore
        w, v = np.linalg.eig(comp_mat)
        weights[j,:] = np.abs(v[:,0])

        # Ensembles distance matrices
        matrix_norms = []
        for i in range(K):
            matrix_norms.append(X_distance_matrix_list[i][:,j]/np.sqrt(np.sum(X_distance_matrix_list[i][:,j]**2)))

        temp = np.zeros(matrix_norms[0].shape)
        for i in range(K):
            temp += matrix_norms[i]*weights[j,i]

        meta_distances[:,j] = temp

    meta_distances = np.nan_to_num((meta_distances+meta_distances.T)/2)

    # Re-project on 2D embedding space
    if projection_method is None:
        return(meta_distances)
    else:
        if projection_method == "tSNE":
            tsne = manifold.TSNE(n_components=2, metric="precomputed", verbose=0).fit(meta_distances)
            meta_visualization = tsne.embedding_
        elif projection_method == "MDS":
            mds = manifold.MDS(n_components=2, verbose=0, dissimilarity="precomputed").fit(meta_distances)
            meta_visualization = mds.embedding_

        return(meta_visualization, meta_distances)
    

def meta_viz_calc(df, random_state=24771):
    model = manifold.LocallyLinearEmbedding(method="standard", n_neighbors=20, random_state=random_state)
    x_embedded_lle=model.fit_transform(df.values)
    model =  manifold.MDS(n_components=2, max_iter=300, n_init=4, random_state=random_state)
    x_embedded_mds = model.fit_transform(df.values)
    model = manifold.SpectralEmbedding(n_components=2, n_neighbors=20, random_state=random_state)
    x_embedded_spc = model.fit_transform(df.values)
    model = PCA(n_components=2, random_state=random_state)
    x_embedded_pca = model.fit_transform(df.values)
    model = manifold.TSNE(n_components=2, perplexity=10, random_state=random_state)
    x_embedded_tsne = model.fit_transform(df.values)
    model = umap.UMAP(n_components=2, random_state=random_state, n_neighbors=10)
    x_embedded_umap = model.fit_transform(df.values)
    meta_visualization, meta_distances = meta_viz([x_embedded_lle, x_embedded_mds, x_embedded_spc, x_embedded_pca, x_embedded_tsne, x_embedded_umap], projection_method="MDS")
    return meta_visualization, meta_distances


def meta_viz_calc_fcs(df, random_state=24771):
    model = manifold.LocallyLinearEmbedding(method="standard", n_neighbors=20, random_state=random_state)
    x_embedded_lle=model.fit_transform(df.values)
    model =  manifold.MDS(n_components=2, max_iter=300, n_init=4, random_state=random_state)
    x_embedded_mds = model.fit_transform(df.values)
    model = manifold.SpectralEmbedding(n_components=2, n_neighbors=20, random_state=random_state)
    x_embedded_spc = model.fit_transform(df.values)
    model = PCA(n_components=2, random_state=random_state)
    x_embedded_pca = model.fit_transform(df.values)
    model = manifold.TSNE(n_components=2, perplexity=20, random_state=random_state)
    x_embedded_tsne = model.fit_transform(df.values)
    model = umap.UMAP(n_components=2, random_state=random_state, n_neighbors=20)
    x_embedded_umap = model.fit_transform(df.values)
    meta_visualization, meta_distances = meta_viz([x_embedded_lle, x_embedded_mds, x_embedded_spc, x_embedded_pca, x_embedded_tsne, x_embedded_umap], projection_method="MDS")
    return meta_visualization, meta_distances


# 第一主成分と第二主成分でプロットする
def METAplot(feature, bacth_lst_ex, batch_c_ex, xnum=0, ynum=1, title="META Visualization", figname=None):
    fig = plt.figure(figsize=(10,7))
    ax=fig.add_subplot(111)
    labelset = []
    ap = labelset.append
    for i in range(len(bacth_lst_ex)):
        if bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]) not in labelset:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            if batch_c_ex[i][1] == 0.8:
                ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, s=70, color="w", linewidth=3, edgecolor=color_map((0.5+0.5)**2), label=bacth_lst_ex[i] + " day" + str(int(10*batch_c_ex[i][1])))
            else:
                ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, s=70, color="w", linewidth=3, edgecolor=color_map(((batch_c_ex[i][1])+0.5)**2), label=bacth_lst_ex[i] + " day" + str(int(10*batch_c_ex[i][1])))
            ap(bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]))
        else:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            if batch_c_ex[i][1] == 0.8:
                ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, s=70, color="w", linewidth=3, edgecolor=color_map((0.5+0.5)**2), label="")
            else:
                ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, s=70, color="w", linewidth=3, edgecolor=color_map(((batch_c_ex[i][1])+0.5)**2), label="")
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), ncol=1)
    ax.set_title(title, fontsize=24)
    ax.set_xlabel("Component "+str(xnum+1), fontsize=20)
    ax.set_ylabel("Component "+str(ynum+1), fontsize=20)

    #ax.set_xlim(-130,410)

    ax.grid(False)
    [ax.spines[side].set_visible(False) for side in ['right','top']]

    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')

    plt.tick_params(bottom=True, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname)
    plt.show()


# 第一主成分と第二主成分でプロットする
def METAplot2(feature, bacth_lst_ex, batch_c_ex, xnum=0, ynum=1, title="META Visualization", figname=None):
    fig = plt.figure(figsize=(9,7))
    ax=fig.add_subplot(111)
    labelset = []
    ap = labelset.append
    for i in range(len(bacth_lst_ex)):
        if bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]) not in labelset:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, s=70, color="w", linewidth=3, edgecolor=color_map((batch_c_ex[i][2])/1000), label=bacth_lst_ex[i] + " day" + str(int(10*batch_c_ex[i][1])))
            ap(bacth_lst_ex[i] + " day" + str(10*batch_c_ex[i][1]))
        else:
            color_map = plt.get_cmap(batch_c_ex[i][0])
            ax.scatter(feature[xnum][i], feature[ynum][i], alpha=1, s=70, color="w", linewidth=3, edgecolor=color_map((batch_c_ex[i][2])/1000), label="")


    a=[-10,-20]
    b=[-10,-20]
    c=[0,1000]
    img = plt.scatter(a,b,c=c,cmap=plt.get_cmap(batch_c_ex[0][0]),alpha=1)
    plt.colorbar(img).set_label("ALT (U/L); values >= 1,000 share maximum color")

    #ax.legend(loc='upper left', bbox_to_anchor=(1, 1), ncol=1)
    ax.set_title(title, fontsize=24)
    ax.set_xlabel("Component "+str(xnum+1), fontsize=20)
    ax.set_ylabel("Component "+str(ynum+1), fontsize=20)

    #ax.set_xlim(-130,410)

    ax.grid(False)
    [ax.spines[side].set_visible(False) for side in ['right','top']]

    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')

    plt.xlim(-0.295,0.2)
    plt.ylim(-0.25,0.25)

    plt.tick_params(bottom=True, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname)
    plt.show()


def low_exp_cut_v4(df, threshold=np.log2(10), count_thresh=3):
    imputer = KNNImputer()
    df_log = np.log2(df+1)
    print("gene count (before processing):" + str(len(df_log)))
    df_log = df_log[df_log>threshold].fillna(float(0)) #threshold以下の値を0に統一
    df_target = df_log[df_log>0].fillna(np.nan) #0をNanに

    # gene selection part
    col_lst = df.columns.tolist()
    print("Number of samples:", len(col_lst))
    df_test = df_target.isnull().sum(axis=1) #遺伝子ごとのNan数の合計を算出
    gene_lst = [ind for ind in df_test.index.tolist() if df_test[ind] > len(col_lst)-count_thresh and df_test[ind] < len(col_lst)] #Nanではない値の数がcount_thresh以下の遺伝子のlistを取得

    up_gene_lst = []
    ap = up_gene_lst.append

    n = len(gene_lst)
    print("Number of genes to be tested:", n)

    for gene in gene_lst:
        fvalue, pvalue = stats.f_oneway(
            df_log.loc[gene, df_log.columns.str.contains("control")], 
            df_log.loc[gene, df_log.columns.str.contains("taa_300 mg/l_24 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("taa_300 mg/l_48 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("taa_300 mg/l_72 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("taa_300 mg/l_96 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("taa_300 mg/l_192 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("mda_750 mg/l_24 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("mda_750 mg/l_48 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("mda_750 mg/l_72 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("mda_750 mg/l_96 hr")], 
            df_log.loc[gene, df_log.columns.str.contains("mda_750 mg/l_192 hr")])
        if pvalue <= 0.05/n:
            #print("up-regulated gene : ", gene)
            ap(gene)
    print("gene count (up-regulated gene):", len(up_gene_lst)) #取得した遺伝子の数
    
    df_log_up_reg = df_log.loc[up_gene_lst, :] #発現上昇 (or 減少)を強調するため0のままのdfを残しておく
    
    df_target = df_target.dropna(thresh=count_thresh) #Nanの数がcount_threshを超えるものは遺伝子そのものを排除
    print("gene count (after dropna):" + str(len(df_target)))

    df_target = pd.DataFrame(imputer.fit_transform(df_target), index=df_target.index.tolist(), columns=df_target.columns.tolist()) #KNN imputerで欠損値の補完

    df_target4 = pd.concat([df_target, df_log_up_reg], axis=0) #事前取得しておいたup-regulated geneをもとに戻す
    print("gene count (after all processing):" + str(len(df_target4)))

    return up_gene_lst, df_target4


def go_selection_v2(go_rank, ref, raito_thresh, go_num=10):
    """
    Extraction of quality GOs
    
    Parameters
    ----------
    go_rank: list
        GO term (sorted)
    
    ref: dict
        {GO:{gene set}, ...}
    
    raito_thresh (default=0.5): float
        Percentage of genes that were overlaped

    go_num (default=10): int
        Number of GOs to be extracted
    
    Returns
    ----------
    good_go_lst: list, str
    
    """

    i = 0
    num = 0
    good_go_lst = []
    ap = good_go_lst.append
    while num < go_num:
        if len(good_go_lst) == 0:
            print(go_rank[i]+" : OK")
            ap(go_rank[i])
            num += 1
            i += 1
        else:
            p = 0
            for go in good_go_lst:
                if len(ref[go]&ref[go_rank[i]])/len(ref[go]) >= raito_thresh or len(ref[go]&ref[go_rank[i]])/len(ref[go_rank[i]]) >= raito_thresh:
                    print(go_rank[i])
                    print("→ this go has elements overlap with many of the top GOs :", go)
                    i += 1
                    p += 1
                    break
                else:
                    pass
            if p == 0:
                print(go_rank[i]+" : OK")
                ap(go_rank[i])
                num += 1
                i += 1

    return good_go_lst


def gage_plot(go, res_lst2, figname=None):
    df_taa = pd.DataFrame()
    df_taa.index = ["comp", "time", "p_val", "q_val", "mean_diff"]

    for i in range(len(res_lst2)):
        if res_lst2[i][1].split("_")[0] == "taa":
            df_taa.loc[:, res_lst2[i][1]] = [res_lst2[i][1].split("_")[0], int(res_lst2[i][1].split("_")[2].split(" ")[0]), res_lst2[i][0].loc[go, "p_val"], res_lst2[i][0].loc[go, "adjusted_p_val"], res_lst2[i][0].loc[go, "mean_diff"]]

    df_taa = df_taa.T.sort_values('time')

    df_mda = pd.DataFrame()
    df_mda.index = ["comp", "time", "p_val", "q_val", "mean_diff"]

    for i in range(len(res_lst2)):
        if res_lst2[i][1].split("_")[0] == "mda":
            df_mda.loc[:, res_lst2[i][1]] = [res_lst2[i][1].split("_")[0], int(res_lst2[i][1].split("_")[2].split(" ")[0]), res_lst2[i][0].loc[go, "p_val"], res_lst2[i][0].loc[go, "adjusted_p_val"], res_lst2[i][0].loc[go, "mean_diff"]]

    df_mda = df_mda.T.sort_values('time')

    plt.style.use('default')

    sns.set()
    sns.set_style('whitegrid')
    sns.set_palette('gist_yarg')

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(1, 1, 1)

    score = "mean_diff"

    ax.plot(df_taa["time"], df_taa[score], color="orangered", linestyle="--", label="TAA", zorder=2)
    ax.plot(df_mda["time"], df_mda[score], color="mediumblue", linestyle="-.", label="MDA", zorder=2)
    ax.legend(fontsize=18)

    for ind in df_taa.index.tolist():
        if math.isnan(df_taa.loc[ind, "p_val"]):
            ax.scatter(df_taa.loc[ind, "time"], df_taa.loc[ind, "mean_diff"], color="orangered", s=-10*np.log2(0.005), marker="x", linewidth=3, edgecolor="orangered", zorder=3)
        
        elif df_taa.loc[ind, "p_val"] == 0:
            ax.scatter(df_taa.loc[ind, "time"], df_taa.loc[ind, "mean_diff"], color="orangered", s=-10*np.log2(df_taa.loc[ind, "p_val"]+0.0000000001), linewidth=1, edgecolor="orangered", marker="*", zorder=3)

        elif df_taa.loc[ind, "q_val"] < 0.05:
            ax.scatter(df_taa.loc[ind, "time"], df_taa.loc[ind, "mean_diff"], color="orangered", s=-10*np.log2(df_taa.loc[ind, "p_val"]), linewidth=1, edgecolor="orangered", zorder=3)
            #ax.text(df_taa.loc[ind, "time"], df_taa.loc[ind, "mean_diff"], "p="+str(round(df_taa.loc[ind, "p_val"], 10)))
        else:
            ax.scatter(df_taa.loc[ind, "time"], df_taa.loc[ind, "mean_diff"], color="white", s=-10*np.log2(0.05), linewidth=1, edgecolor="orangered", zorder=3)

    for ind in df_mda.index.tolist():
        if math.isnan(df_mda.loc[ind, "p_val"]):
            ax.scatter(df_mda.loc[ind, "time"], df_mda.loc[ind, "mean_diff"], color="mediumblue", s=-10*np.log2(0.005), marker="x", linewidth=3, edgecolor="mediumblue", zorder=3)

        elif df_mda.loc[ind, "p_val"] == 0:
            ax.scatter(df_mda.loc[ind, "time"], df_mda.loc[ind, "mean_diff"], color="mediumblue", s=-10*np.log2(df_mda.loc[ind, "p_val"]+0.0000000001), linewidth=1, edgecolor="mediumblue", marker="*", zorder=3)

        elif df_mda.loc[ind, "q_val"] < 0.05:
            ax.scatter(df_mda.loc[ind, "time"], df_mda.loc[ind, "mean_diff"], color="mediumblue", s=-10*np.log2(df_mda.loc[ind, "p_val"]), linewidth=1, edgecolor="mediumblue", zorder=3)
            #ax.text(df_mda.loc[ind, "time"], df_mda.loc[ind, "mean_diff"], "p="+str(round(df_mda.loc[ind, "p_val"], 10)))
        else:
            ax.scatter(df_mda.loc[ind, "time"], df_mda.loc[ind, "mean_diff"], color="white", s=-10*np.log2(0.05), linewidth=1, edgecolor="mediumblue", zorder=3)

    ax.hlines(y=0, xmin=0, xmax=200, color="black", linestyles='dotted', alpha=.8, zorder=1) # threshold line

    ax.grid(False)
    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')

    ax.set_xlim(0,200)

    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18, rotation=0)
    ax.set_xlabel('time (h)', fontsize=22)
    ax.set_ylabel(score, fontsize=22)

    plt.tick_params(bottom=False, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    if len(" ".join(go.split("_")[1:])) > 29:
        if len(go.split("_")[1:])%2 == 1:
            n = int(len(go.split("_")[1:])//2 + 1)
        else:
            n = int(len(go.split("_")[1:])/2)
        fig.suptitle(" ".join(go.split("_")[1:n+1])+"\n"+str(" ".join(go.split("_")[n+1:])), fontsize=24)

    else:
        fig.suptitle("\n"+" ".join(go.split("_")[1:]), fontsize=24)

    plt.tick_params(bottom=False, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname)
    plt.show()

## = Model construction = ##

### ALT & AST ###

In [ ]:
df = pd.read_csv(f"{data_root}/raw/biochemistry/drink_TAA_MDA_chronic.csv")
df = df.iloc[0:33, :]
df_ALT = df.loc[:, df.columns.str.contains("ALT")]
df_AST = df.loc[:, df.columns.str.contains("AST")]
df_ALT = df_ALT.rename(columns={'ALT':'day7', 'ALT.1':'day14', 'ALT.2':'day28'})
df_AST = df_AST.rename(columns={'AST':'day7', 'AST.1':'day14', 'AST.2':'day28'})
# The time-course and Table 1 use only animals followed through Day 28.
assert df.loc[df["ALT.2"].notna(), "group"].value_counts().to_dict() == {"TAA": 6, "MDA": 6, "Control": 4}
assert pd.isna(df.loc[df["No."] == 32, "AST.1"]).all()

In [ ]:
plot_line(df_ALT, figname=f"{fig_root}/fig_s2.png")

In [ ]:
one_way_anova(df_ALT, day="day7", val_name="ALT")

In [ ]:
one_way_anova(df_AST, day="day7", val_name="AST")

In [ ]:
one_way_anova(df_ALT, day="day14", val_name="ALT")

In [ ]:
one_way_anova(df_AST, day="day14", val_name="AST")

In [ ]:
one_way_anova(df_ALT, day="day28", val_name="ALT")

In [ ]:
one_way_anova(df_AST, day="day28", val_name="AST")

### qPCR ###

In [ ]:
# day14
df = pd.read_csv(f"{data_root}/raw/qpcr/231020_dw_day14_fib_2309sample_v2 -  Quantification Cq Results.csv", index_col=0)
df_sq = calc_p(df, internal_standard="GAPDH")

# day28
df2 = pd.read_csv(f"{data_root}/raw/qpcr/231023_dw_day28_fib_2309sample -  Quantification Cq Results.csv", index_col=0)
df_sq2 = calc_p(df2, internal_standard="GAPDH")

# ウェルチ、両側、多重検定補正なし、

In [ ]:
sq_plot([df_sq, df_sq2], internal_standard="GAPDH", target_gene="Acta2", y_max=4.0, figname=f"{fig_root}/fig_1a.png")

In [ ]:
sq_plot([df_sq, df_sq2], internal_standard="GAPDH", target_gene="Col1a1", y_max=1.5, figname=f"{fig_root}/fig_1b.png")

## = Cohort 1 = ##

### ALT & AST ###

In [ ]:
df_total = pd.read_csv(f"{data_root}/raw/biochemistry/240111_DrinkingWaterModel_rawdata_v6.csv", index_col=0)

In [ ]:
value_plot(df_total, target="ALT", comp_lst=["Control", "TAA", "MDA"], c_lst=["gray", "coral", "royalblue"], title="ALT (U/L)", figname=f"{fig_root}/fig_s3_1.png")

In [ ]:
value_plot(df_total, target="AST", comp_lst=["Control", "TAA", "MDA"], c_lst=["gray", "coral", "royalblue"], title="AST (U/L)", figname=f"{fig_root}/fig_s3_2.png")

### ALP, TBIL, TCHO, ###

In [ ]:
value_plot(df_total, target="ALP", comp_lst=["Control", "TAA", "MDA"], c_lst=["gray", "coral", "royalblue"], title="ALP (U/l)", figname=f"{fig_root}/fig_s3_3.png")

In [ ]:
value_plot(df_total, target="TBIL", comp_lst=["Control", "TAA", "MDA"], c_lst=["gray", "coral", "royalblue"], title="TBIL (mg/dl)", figname=f"{fig_root}/fig_s3_4.png")

In [ ]:
value_plot(df_total, target="TCHO", comp_lst=["Control", "TAA", "MDA"], c_lst=["gray", "coral", "royalblue"], title="TCHO (mg/dl)", figname=f"{fig_root}/fig_s3_5.png")

In [ ]:
value_plot(df_total, target="HDL", comp_lst=["Control", "TAA", "MDA"], c_lst=["gray", "coral", "royalblue"], title="HDL (mg/dl)", figname=f"{fig_root}/fig_s3_6.png")

In [ ]:
value_plot(df_total, target="TG", comp_lst=["Control", "TAA", "MDA"], c_lst=["gray", "coral", "royalblue"], title="TG (mg/dl)", figname=f"{fig_root}/fig_s3_7.png")

## = facs = ##

### Violin plot of each cell ###

In [ ]:
"""
Significance annotation for the immune-cell violin plots (Supplementary Fig. 4).

Two pieces:

1. compute_immune_stats(...)  — runs the whole test family once and returns a
   lookup table. It must see every population and every day, because the
   Benjamini-Hochberg correction is applied across all 55 Kruskal-Wallis tests
   (11 populations x 5 time points), not within a single panel.

2. violinplot_twogroup(...)   — draws the figure and adds significance annotations
   from the lookup table.

Usage
-----
    stats_all = compute_immune_stats(df_all, populations=POPS)   # df_all includes Control
    violinplot_twogroup(df_dili, immune="CD8T", celltype="lymph",
                        stats_result=stats_all, figname=...)

Dependency: scikit-posthocs (pinned in environment/requirements-lock.txt)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sph


# --------------------------------------------------------------------------
# statistics
# --------------------------------------------------------------------------

def compute_immune_stats(df_all, populations, group_col="group", day_col="day",
                         groups=("Control", "MDA", "TAA"), alpha=0.05):
    """
    Run the full test family over every population x time point.

    Returns
    -------
    dict: stats[population][day] = {"kw_p", "kw_q", "significant", "pairs"}
          where "pairs" maps frozenset({g1, g2}) -> Dunn p-value, and is empty
          when the BH-corrected Kruskal-Wallis p-value was not below alpha.
    """
    days = sorted(df_all[day_col].unique())

    # --- omnibus test -----------------------------------------------------
    # Kruskal-Wallis: the rank-based counterpart of the one-way ANOVA used for
    # the plasma biochemical data. Chosen because these values are proportions,
    # so the variance scales with the mean and differs by more than two orders
    # of magnitude between populations, and because the small, unequal group
    # sizes (n = 3-8) do not permit verification of normality or homogeneity of
    # variance.
    rows = []
    for pop in populations:
        for day in days:
            sub = df_all[df_all[day_col] == day]
            samples = [sub.loc[sub[group_col] == g, pop].dropna().values
                       for g in groups]
            H, p = sp_stats.kruskal(*samples)
            rows.append({"pop": pop, "day": day, "H": H, "p": p})
    kw = pd.DataFrame(rows)

    # --- correction across the whole family --------------------------------
    # Benjamini-Hochberg over all 11 populations x 5 time points (55 tests),
    # not within a panel, because the populations were screened together.
    kw["q"] = multipletests(kw["p"].values, method="fdr_bh")[1]

    # --- post hoc ----------------------------------------------------------
    # Dunn's test, the rank-based counterpart of Tukey's HSD, with Holm
    # correction over the three pairwise comparisons within each test. Run only
    # where the corrected omnibus p-value was below alpha, so that the post hoc
    # comparisons are conditioned on a significant Kruskal-Wallis result.
    out = {pop: {} for pop in populations}
    for _, r in kw.iterrows():
        entry = {"kw_p": r["p"], "kw_q": r["q"],
                 "significant": bool(r["q"] < alpha), "pairs": {}}
        if entry["significant"]:
            sub = df_all[df_all[day_col] == r["day"]][[group_col, r["pop"]]].dropna()
            sub.columns = ["group", "value"]
            m = sph.posthoc_dunn(sub, val_col="value", group_col="group",
                                 p_adjust="holm")
            for i, g1 in enumerate(groups):
                for g2 in groups[i + 1:]:
                    entry["pairs"][frozenset((g1, g2))] = float(m.loc[g1, g2])
        out[r["pop"]][r["day"]] = entry
    return out


def _stars(p, ns_label=""):
    """p-value to symbol, matching the convention used elsewhere in the paper."""
    if p is None or np.isnan(p):
        return ns_label
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ns_label


# --------------------------------------------------------------------------
# plot
# --------------------------------------------------------------------------

def _symbols(p, symbol="*", ns_label=""):
    """p-value to a repeated symbol, so that vs-control and between-group
    comparisons can be distinguished by glyph as well as by position."""
    if p is None or np.isnan(p):
        return ns_label
    if p < 0.001:
        return symbol * 3
    if p < 0.01:
        return symbol * 2
    if p < 0.05:
        return symbol
    return ns_label


def violinplot_twogroup_v2(df_dili, immune='', celltype='', figname=None,
                           stats_result=None, control_label="Control",
                           ns_label="", annot_fontsize=14,
                           control_symbol="*", between_symbol="#"):
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(1, 1, 1)

    flatui = ["coral", "royalblue"]
    sns.set_palette(flatui)
    sns.violinplot(data=df_dili, x='day', y=immune, inner=None, hue='group', split=True)

    if celltype == "lymph":
        fig.legend(frameon=True, loc=(0.755, 0.75), fontsize=10, framealpha=1)
    elif celltype == "mye":
        fig.legend(frameon=True, loc=(0.25, 0.75), fontsize=10, framealpha=1)

    flatui = ["orangered", "mediumblue"]
    sns.set_palette(flatui)
    # Draw every non-missing animal at deterministic, distinct x positions.
    day_levels = list(pd.unique(df_dili['day']))
    hue_levels = list(pd.unique(df_dili['group']))
    point_colors = dict(zip(hue_levels, flatui))
    for xi, day in enumerate(day_levels):
        for gi, group in enumerate(hue_levels):
            vals = df_dili.loc[(df_dili['day'] == day) &
                               (df_dili['group'] == group), immune].dropna().to_numpy()
            offsets = np.linspace(-0.06, 0.06, len(vals)) if len(vals) > 1 else np.zeros(len(vals))
            center = xi + (-0.13 if gi == 0 else 0.13)
            ax.scatter(center + offsets, vals, s=25, c=point_colors[group],
                       edgecolors="black", linewidths=1, zorder=3)

    ax.get_legend().remove()

    plt.grid(False)
    plt.tick_params(bottom=True, left=True, right=False, top=False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    ax.spines.bottom.set_color('black')
    ax.spines.left.set_color('black')
    ax.set_title(immune, fontsize=24)
    ax.set_xlabel("day", fontsize=20)
    ax.set_ylabel("population", fontsize=20)
    ax.tick_params(axis='x', labelsize=16)
    ax.tick_params(axis='y', labelsize=16)

    # ---- significance annotation ----------------------------------------
    # Two kinds of comparison are drawn, distinguished both by glyph and by
    # vertical position:
    #   control_symbol (#)  each plotted group vs the concurrent control,
    #                       placed over its own half of the split violin
    #   between_symbol (*)  MDA vs TAA, drawn as a bracket above them
    if stats_result is not None and immune in stats_result:
        days_plotted = [t.get_text() for t in ax.get_xticklabels()]
        hue_levels = list(pd.unique(df_dili['group']))          # split order: left, right
        ymin, ymax = ax.get_ylim()
        span = ymax - ymin
        top = ymax                                              # tracks the highest annotation

        for xi, day_label in enumerate(days_plotted):
            # x tick labels are strings; match them back to the keys used in stats
            day_key = next((d for d in stats_result[immune]
                            if str(d) == day_label), None)
            if day_key is None:
                continue
            entry = stats_result[immune][day_key]
            if not entry["significant"]:
                continue

            # height reference: top of the observed values on this day
            vals = df_dili.loc[df_dili['day'].astype(str) == day_label, immune].dropna()
            base = float(vals.max()) if len(vals) else ymin
            step = span * 0.09          # annotation spacing

            # --- vs concurrent control, one per half -------------------------
            for k, g in enumerate(hue_levels):
                p = entry["pairs"].get(frozenset((control_label, g)))
                lab = _symbols(p, control_symbol, ns_label)
                if lab:
                    y = base + step * 0.35
                    ax.text(xi + (-0.20 if k == 0 else 0.20), y, lab,
                            ha="center", va="bottom",
                            fontsize=annot_fontsize, clip_on=False)
                    top = max(top, y + step * 0.5)

            # --- MDA vs TAA, as a bracket well clear of the symbols ----------
            if len(hue_levels) == 2:
                p = entry["pairs"].get(frozenset(tuple(hue_levels)))
                lab = _symbols(p, between_symbol, ns_label)
                if lab:
                    y = base + step * 1.6
                    h = span * 0.018
                    ax.plot([xi - 0.20, xi - 0.20, xi + 0.20, xi + 0.20],
                            [y, y + h, y + h, y], lw=1.0, c="black",
                            clip_on=False)
                    ax.text(xi, y + h * 1.1, lab, ha="center", va="bottom",
                            fontsize=annot_fontsize, clip_on=False)
                    top = max(top, y + h + step * 0.6)

        ax.set_ylim(ymin, max(ymax, top))
    # ---------------------------------------------------------------------

    plt.tight_layout()
    if figname:
        save_panel(plt.gcf(), figname, bbox_inches="tight")
    plt.show()

def stats_to_df(stats_result):
    rows = []
    for pop, per_day in stats_result.items():
        for day, entry in per_day.items():
            base = {
                "population": pop,
                "day": day,
                "kw_p": entry["kw_p"],
                "kw_q_BH": entry["kw_q"],
                "kw_significant": entry["significant"],
            }
            if entry["pairs"]:
                for pair, p in entry["pairs"].items():
                    g1, g2 = sorted(pair)
                    rows.append({**base,
                                 "comparison": f"{g1} vs {g2}",
                                 "dunn_p_holm": p,
                                 "dunn_significant": p < 0.05})
            else:
                rows.append({**base, "comparison": None,
                             "dunn_p_holm": None, "dunn_significant": None})
    df = pd.DataFrame(rows)
    return df.sort_values(["population", "day", "comparison"]).reset_index(drop=True)

In [ ]:
flatui = ["orangered", "mediumblue"]
sns.set_palette(flatui)
sns.color_palette(n_colors=2)

In [ ]:
df_total = pd.read_csv(f"{data_root}/raw/facs/230605_data_blood_facs.csv", index_col=0)
col_lst = []
ap = col_lst.append
for col in df_total.columns.tolist():
    if col == "neutrophil" or col == "eosinophil":
        ap(col.capitalize())
    else:
        ap(col)
df_total.columns = col_lst
df_dili = df_total[~df_total.loc[:, "group"].str.contains("Control")]

In [ ]:
POPS = ['gdT','NKT','NK','CD4T','CD8T','B','Monocyte',
        'Neutrophil','Kupffer','Eosinophil','MonMac']

stats_all = compute_immune_stats(df_total, populations=POPS)

In [ ]:
df_stats = stats_to_df(stats_all)
df_stats.to_csv(f"{table_root}/immune_stats.csv", index=False)

In [ ]:
target = [
 'CD4T',
 'Neutrophil',
 'Eosinophil',
 'CD8T'
 ]
for n, i in enumerate(target):
    fig_name = "fig_4b_"+str(n+1)+".png"
    name = f"{fig_root}/{fig_name}"
    if i == "CD4T":
        violinplot_twogroup_v2(df_dili, immune=i, celltype='lymph', figname=name, stats_result=stats_all)
    else:
        violinplot_twogroup_v2(df_dili, immune=i, celltype='mye', figname=name, stats_result=stats_all)

In [ ]:
target = [
 'gdT',
 'NKT',
 'NK',
 'B',
 'Monocyte',
 'MonMac',
 'Kupffer',
 ]
for n, i in enumerate(target):
    fig_name = "fig_s4_"+str(n+1)+".png"
    name = f"{fig_root}/{fig_name}"
    if i == "Monocyte":
        violinplot_twogroup_v2(df_dili, immune=i, celltype='mye', figname=name, stats_result=stats_all)
    elif i == "NK":
        violinplot_twogroup_v2(df_dili, immune=i, celltype='mye', figname=name, stats_result=stats_all)
    elif i == "MonMac":
        violinplot_twogroup_v2(df_dili, immune=i, celltype='mye', figname=name, stats_result=stats_all)
    else:
        violinplot_twogroup_v2(df_dili, immune=i, celltype='lymph', figname=name, stats_result=stats_all)

### meta viz ###

In [ ]:
df_total = pd.read_csv(f"{data_root}/raw/facs/230605_data_blood_facs.csv", index_col=0)
col_lst = []
ap = col_lst.append
for col in df_total.columns.tolist():
    if col == "neutrophil" or col == "eosinophil":
        ap(col.capitalize())
    else:
        ap(col)
df_total.columns = col_lst

In [ ]:
imu_lst = [
 'gdT',
 'NKT',
 'NK',
 'CD4T',
 'CD8T',
 'B',
 'Monocyte',
 'Neutrophil',
 'Kupffer',
 'Eosinophil',
 'MonMac'
 ]

In [ ]:
df_total.index = df_total.loc[:, "ID"]
df_fcs = df_total.loc[:, imu_lst+["day"]]
df_fcs = df_fcs.dropna(how='any')

In [ ]:
bacth_lst_comp = []
ap = bacth_lst_comp.append
for i in df_fcs.index:
    if "C" in i:
        ap("Control")
    elif "T" in i:
        ap("TAA")
    elif "M" in i:
        ap("MDA")
dct = {"Control": "Greys", "TAA": "Oranges", "MDA": "Blues"}

In [ ]:
batch_c_comp = [dct[i] for i in bacth_lst_comp]
day_lst = [float(df_fcs.iloc[i, 11])/10 for i in range(len(df_fcs))]
batch_c_comp_2 = [[dct[j], day_lst[i]] for i, j in enumerate(bacth_lst_comp)]

In [ ]:
df_fcs = df_fcs.loc[:, imu_lst]
# Standardization
dfs = df_fcs.apply(lambda x: (x-x.mean())/x.std(), axis=0)
dfs_fix = dfs.dropna(axis=1)

In [ ]:
pca = PCA()
pca.fit(dfs_fix)
feature = pca.transform(dfs_fix)

In [ ]:
PCAplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="PCA")

In [ ]:
df_target = np.log2(df_fcs+1)
#標準化
dfs = df_target .apply(lambda x: (x-x.mean())/x.std(), axis=0)

print("len(dfs) = ", len(dfs.T))
dfs_fix = dfs.dropna(how='all', axis=1)
print("len(dfs_fix) = ", len(dfs_fix.T))

pca = PCA()
pca.fit(dfs_fix)
feature_log = pca.transform(dfs_fix)

In [ ]:
PCAplot(feature_log, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="PCA (log)")

In [ ]:
model =  manifold.MDS(n_components=2, max_iter=300, n_init=4, random_state=24771) # max_iter, n_initは初期値
x_embedded_mds = model.fit_transform(dfs_fix.values)
feature = [[i[0] for i in x_embedded_mds], [i[1]for i in x_embedded_mds]]

In [ ]:
tSNEplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="MDS")

In [ ]:
for n, i in enumerate([5, 10, 20, 30]):
    model_tsne = TSNE(n_components=2, perplexity=i, random_state=24771)
    tsne_feature = model_tsne.fit_transform(dfs_fix)
    feature = [[i[0] for i in tsne_feature], [i[1]for i in tsne_feature]]
    tSNEplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="t-SNE, perplexity="+str(i))

In [ ]:
for n, i in enumerate([5, 10, 20, 30]):
    model_umap = umap.UMAP(n_components=2, random_state=24771, n_neighbors=i)
    umap_feature = model_umap.fit_transform(dfs_fix)
    feature = [[i[0] for i in umap_feature], [i[1]for i in umap_feature]]
    UMAPplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="UMAP, n_neighbors="+str(i))

In [ ]:
for n, i in enumerate([5, 10, 20, 30, 50]):
    model = manifold.SpectralEmbedding(n_components=2, n_neighbors=i, random_state=24771)
    x_embedded_spc = model.fit_transform(dfs_fix.values)
    feature = [[i[0] for i in x_embedded_spc], [i[1]for i in x_embedded_spc]]
    UMAPplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="spc, n_neighbors="+str(i))

In [ ]:
for n, i in enumerate([5, 10, 20, 30, 50]):
    model = manifold.LocallyLinearEmbedding(method="standard", n_neighbors=i, random_state=24771)
    x_embedded_lle=model.fit_transform(dfs_fix.values)
    feature = [[i[0] for i in x_embedded_lle], [i[1]for i in x_embedded_lle]]
    UMAPplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="LLE, n_neighbors="+str(i))

In [ ]:
import warnings
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams["font.size"] = 18
warnings.simplefilter('ignore')

In [ ]:
meta_visualization, meta_distances = meta_viz_calc_fcs(dfs_fix)

In [ ]:
feature_meta = [[meta_visualization[i][0] for i in range(len(meta_visualization))], [meta_visualization[i][1] for i in range(len(meta_visualization))]]

In [ ]:
METAplot(feature_meta, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, figname=f"{fig_root}/fig_4a.png")

## = META Viz = ##

### support ###

In [ ]:
df = pd.read_csv(f"{data_root}/raw/rnaseq/res_annotation.csv", index_col=0)
df_raw = pd.read_csv(f"{data_root}/raw/rnaseq/230528_DrinkingWaterModel_rawdata_v2.csv", index_col=0).iloc[:, 0:14]
clm_lst = [df_raw.iloc[i, 1]+"_"+str(df_raw.iloc[i, 2])+" mg/l_"+str(24*df_raw.iloc[i, 3])+" hr."+df_raw.iloc[i, 0] for i in range(len(df_raw)) if int(df_raw.iloc[i, 12]) != 1]
df.columns = clm_lst
df = df.T

In [ ]:
bacth_lst_comp = [df_raw.iloc[i, 1] for i in range(len(df_raw)) if int(df_raw.iloc[i, 12]) != 1]
dct = {"Control": "Greys", "TAA": "Oranges", "MDA": "Blues"}
batch_c_comp = [dct[i] for i in bacth_lst_comp]
day_lst = [float(df_raw.iloc[i, 3])/10 for i in range(len(df_raw)) if int(df_raw.iloc[i, 12]) != 1]
batch_c_comp_2 = [[dct[j], day_lst[i]] for i, j in enumerate(bacth_lst_comp)]

# Standardization
dfs = df.apply(lambda x: (x-x.mean())/x.std(), axis=0)
dfs_fix = dfs.dropna(axis=1)

In [ ]:
pca = PCA()
pca.fit(dfs_fix)
feature = pca.transform(dfs_fix)

In [ ]:
PCAplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="PCA")

In [ ]:
df_target = np.log2(df+1)
#標準化
dfs = df_target .apply(lambda x: (x-x.mean())/x.std(), axis=0)

print("len(dfs) = ", len(dfs.T))
dfs_fix = dfs.dropna(how='all', axis=1)
print("len(dfs_fix) = ", len(dfs_fix.T))

pca = PCA()
pca.fit(dfs_fix)
feature_log = pca.transform(dfs_fix)

In [ ]:
PCAplot(feature_log, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="PCA (log)", figname=f"{fig_root}/fig_s5_1.png")

In [ ]:
model =  manifold.MDS(n_components=2, max_iter=300, n_init=4, random_state=24771) # max_iter, n_initは初期値
x_embedded_mds = model.fit_transform(dfs_fix.values)
feature = [[i[0] for i in x_embedded_mds], [i[1]for i in x_embedded_mds]]

In [ ]:
tSNEplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="MDS", figname=f"{fig_root}/fig_s5_2.png")

In [ ]:
for n, i in enumerate([5, 10, 20, 30]):
    if i == 10:
        fig_name = "fig_s5_3.png"
        name = f"{fig_root}/{fig_name}"
    else:
        name = None
    model_tsne = TSNE(n_components=2, perplexity=i, random_state=24771)
    tsne_feature = model_tsne.fit_transform(dfs_fix)
    feature = [[i[0] for i in tsne_feature], [i[1]for i in tsne_feature]]
    tSNEplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="t-SNE, perplexity="+str(i), figname=name)

In [ ]:
for n, i in enumerate([5, 10, 20, 30]):
    if i == 10:
        fig_name = "fig_s5_4.png"
        name = f"{fig_root}/{fig_name}"
    else:
        name = None
    model_umap = umap.UMAP(n_components=2, random_state=24771, n_neighbors=i)
    umap_feature = model_umap.fit_transform(dfs_fix)
    feature = [[i[0] for i in umap_feature], [i[1]for i in umap_feature]]
    UMAPplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="UMAP, n_neighbors="+str(i), figname=name)

In [ ]:
model = manifold.SpectralEmbedding(n_components=2, n_neighbors=20, random_state=24771)
x_embedded_spc = model.fit_transform(dfs_fix.values)
feature = [[i[0] for i in x_embedded_spc], [i[1]for i in x_embedded_spc]]

In [ ]:
for n, i in enumerate([5, 10, 20, 30]):
    if i == 20:
        fig_name = "fig_s5_5.png"
        name = f"{fig_root}/{fig_name}"
    else:
        name = None
    model = manifold.SpectralEmbedding(n_components=2, n_neighbors=i, random_state=24771)
    x_embedded_spc = model.fit_transform(dfs_fix.values)
    feature = [[i[0] for i in x_embedded_spc], [i[1]for i in x_embedded_spc]]
    UMAPplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="spc, n_neighbors="+str(i), figname=name)

In [ ]:
model = manifold.LocallyLinearEmbedding(method="standard", random_state=24771)
x_embedded_lle=model.fit_transform(dfs_fix.values)
feature = [[i[0] for i in x_embedded_lle], [i[1]for i in x_embedded_lle]]

In [ ]:
for n, i in enumerate([5, 10, 20, 30]):
    if i == 20:
        fig_name = "fig_s5_6.png"
        name = f"{fig_root}/{fig_name}"
    else:
        name = None
    model = manifold.LocallyLinearEmbedding(method="standard", n_neighbors=i, random_state=24771)
    x_embedded_lle=model.fit_transform(dfs_fix.values)
    feature = [[i[0] for i in x_embedded_lle], [i[1]for i in x_embedded_lle]]
    UMAPplot(feature, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, title="LLE, n_neighbors="+str(i), figname=name)

### main ###

In [ ]:
import warnings
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams["font.size"] = 18
warnings.simplefilter('ignore')

In [ ]:
df = pd.read_csv(f"{data_root}/raw/rnaseq/res_annotation.csv", index_col=0)
df_raw = pd.read_csv(f"{data_root}/raw/rnaseq/230528_DrinkingWaterModel_rawdata_v2.csv", index_col=0).iloc[:, 0:14]
clm_lst = [df_raw.iloc[i, 1]+"_"+str(df_raw.iloc[i, 2])+" mg/l_"+str(24*df_raw.iloc[i, 3])+" hr."+df_raw.iloc[i, 0] for i in range(len(df_raw)) if int(df_raw.iloc[i, 12]) != 1]
df.columns = clm_lst
df = df.T

In [ ]:
df_target = np.log2(df+1)
#標準化
dfs = df_target .apply(lambda x: (x-x.mean())/x.std(), axis=0)

print("len(dfs) = ", len(dfs.T))
dfs_fix = dfs.dropna(how='all', axis=1)
print("len(dfs_fix) = ", len(dfs_fix.T))

In [ ]:
meta_visualization, meta_distances = meta_viz_calc(dfs_fix)

In [ ]:
bacth_lst_comp = [df_raw.iloc[i, 1] for i in range(len(df_raw)) if int(df_raw.iloc[i, 12]) != 1]
dct = {"Control": "Greys", "TAA": "Oranges", "MDA": "Blues"}
batch_c_comp = [dct[i] for i in bacth_lst_comp]
day_lst = [float(df_raw.iloc[i, 3])/10 for i in range(len(df_raw)) if int(df_raw.iloc[i, 12]) != 1]
batch_c_comp_2 = [[dct[j], day_lst[i]] for i, j in enumerate(bacth_lst_comp)]

In [ ]:
feature_meta = [[meta_visualization[i][0] for i in range(len(meta_visualization))], [meta_visualization[i][1] for i in range(len(meta_visualization))]]

In [ ]:
METAplot(feature_meta, bacth_lst_comp, batch_c_comp_2, xnum=0, ynum=1, figname=f"{fig_root}/fig_5a.png")

In [ ]:
# Match ALT to the exact RNA-seq sample IDs, without relying on row order.
meta_early = pd.read_csv(f"{data_root}/raw/integrated/2610_all_data.csv")
assert meta_early["ID"].is_unique
meta_early = meta_early.set_index("ID")
kept_ids = df_raw.loc[df_raw["to_be_removed"] != 1, "ID"].astype(str)
assert kept_ids.is_unique and len(kept_ids) == len(bacth_lst_comp)
assert kept_ids.isin(meta_early.index).all()
actual_rna_ids = pd.read_csv(f"{data_root}/raw/rnaseq/res_annotation.csv", nrows=0).columns[1:].tolist()
assert meta_early.loc[kept_ids, "rnaseq_file"].tolist() == actual_rna_ids
alt_lst = meta_early.loc[kept_ids, "ALT"].astype(float).tolist()
batch_c_comp_3 = [["jet", day_lst[i], alt_lst[i]] for i in range(len(bacth_lst_comp))]

In [ ]:
METAplot2(feature_meta, bacth_lst_comp, batch_c_comp_3, xnum=0, ynum=1, figname=f"{fig_root}/fig_5b.png")

## = GAGE = ##

### main ###

In [ ]:
df = pd.read_csv(f"{data_root}/raw/rnaseq/res_annotation.csv", index_col=0)
df_raw = pd.read_csv(f"{data_root}/raw/rnaseq/230528_DrinkingWaterModel_rawdata_v2.csv", index_col=0).iloc[:, 0:14]

# 小文字への変換
ind_lst = [i.lower() for i in df.index.tolist()]
clm_lst = [(df_raw.iloc[i, 1]+"_"+str(df_raw.iloc[i, 2])+" mg/l_"+str(24*df_raw.iloc[i, 3])+" hr."+df_raw.iloc[i, 0]).lower() for i in range(len(df_raw)) if int(df_raw.iloc[i, 12]) != 1]
df.index = ind_lst
df.columns = clm_lst
ref = pd.read_pickle(f"{data_root}/references/go_mouse_msigdb.pickle")
for i in ref.keys():
    ref[i] = {j.lower() for j in ref[i]}

In [ ]:
gene_lst = []
apgene = gene_lst.append
for geneset in list(ref.values()):
    for gene in geneset:
        apgene(gene)
gene_set = set(gene_lst)
gene_lst = list(gene_set)
print("total gene count (from go_mouse_msigdb):", len(gene_lst))
print("total gene count (from rnaseq data):", len(df.index.tolist()))

overlap_gene_lst = list(set(df.index.tolist()) & gene_set)
print("overlap gene count :", len(overlap_gene_lst ))

In [ ]:
df_target = df.loc[overlap_gene_lst, :]
df_log = np.log2(df_target+1)

In [ ]:
# Confirm distribution

for i in range(72):
    plt.hist(df_log.iloc[:,i],bins=200,alpha=0.65, range=(0, 20)) # Insert numbers as you see fit
plt.show()

In [ ]:
up_gene_lst, df_target4 = low_exp_cut_v4(df_target, threshold=np.log2(1), count_thresh=50) 

In [ ]:
# Confirm distribution

for i in range(72):
    plt.hist(df_target4.iloc[:,i],bins=200,alpha=0.65, range=(0, 20)) # Insert numbers as you see fit
plt.show()

In [ ]:
gene_lst = df_target4.index.tolist()

In [ ]:
ratio = 0
num = 15
print("go count (before processing): " + str(len(ref)))
n = 0
for i in list(ref.keys()):
    if len(ref[i] & set(gene_lst)) <= num:
        n += 1
        del ref[i]
    elif len(ref[i] & set(gene_lst)) < ratio*len(ref[i]):
        print(ref[i])
        n += 1
        del ref[i]
print("go count (After processing): " + str(len(ref)))
print("go count (del): " + str(n))

In [ ]:
res_lst_v4 = []
ap = res_lst_v4.append
target_lst = list(set([i.split(".")[0] for i in df_target4.columns.tolist() if i.split("_")[0] != "control"]))
for group in target_lst:
    res = ting.calc(
    data=df_target4,
    treatment=group,
    control="control",
    group=ref,
    sign=True,
    )
    ls = [res, group]
    ap(ls)
    print(group + " end")

In [ ]:
df_gage_raw = res_lst_v4[1][0]
df_gage_raw["ID"] = [res_lst_v4[1][1]]*len(df_gage_raw)

In [ ]:
gage_rawdata = pd.DataFrame()

for lst in res_lst_v4:
    df_gage_raw = lst[0]
    df_gage_raw["ID"] = [lst[1]]*len(df_gage_raw)
    gage_rawdata = pd.concat([gage_rawdata, df_gage_raw])

In [ ]:
gage_rawdata.columns = ["p_val", "adjusted_p_val", "corrected_negative_log_sum", "mean_diff", "mean_t", "condition"]

In [ ]:
gage_rawdata.to_csv(f"{table_root}/gage_rawdata.csv")

In [ ]:
mda_meandiff = pd.DataFrame()
taa_meandiff = pd.DataFrame()

for i in range(len(res_lst_v4)):
    if res_lst_v4[i][1].split("_")[0] == "mda":
        #print(res_lst_v4[i][1])
        for go in res_lst_v4[i][0].index.tolist():
            mda_meandiff.loc[go, int(res_lst_v4[i][1].split("_")[2].split(" ")[0])] = res_lst_v4[i][0].loc[go, "mean_diff"]
    
    if res_lst_v4[i][1].split("_")[0] == "taa":
        #print(res_lst_v4[i][1])
        for go in res_lst_v4[i][0].index.tolist():
            taa_meandiff.loc[go, int(res_lst_v4[i][1].split("_")[2].split(" ")[0])] = res_lst_v4[i][0].loc[go, "mean_diff"]

In [ ]:
mda_meandiff = mda_meandiff.sort_index(axis=1, ascending=True)
taa_meandiff = taa_meandiff.sort_index(axis=1, ascending=True)

In [ ]:
res_mda = rp.calc(
    data=mda_meandiff.iloc[:, 0:3]
    )
res_taa = rp.calc(
    data=taa_meandiff.iloc[:, 0:3]
    )

In [ ]:
res_taa.head(20)

In [ ]:
res_mda.head(20)

In [ ]:
gene_lst = df_target4.index.tolist()
go_rank_taa = res_taa.index.tolist()
go_rank_mda = res_mda.index.tolist()

In [ ]:
golst_taa = go_selection_v2(go_rank_taa, ref, raito_thresh=0.5, go_num=20)

In [ ]:
golst_mda = go_selection_v2(go_rank_mda, ref, raito_thresh=0.5, go_num=20)

In [ ]:
go_lst_pickup = ['GOBP_RESPONSE_TO_INTERLEUKIN_17', 'GOBP_FIBRINOLYSIS']
for n, go in enumerate(go_lst_pickup):
    fig_name = "fig_5c_"+["1", "2"][n]+".png"
    name = f"{fig_root}/{fig_name}"
    gage_plot(go, res_lst_v4, figname=name)

## = Water = ##

In [ ]:
df_water = pd.read_csv(f"{data_root}/raw/drinking_water/waterdata.csv", index_col=0)
df_water

In [ ]:
def plot_line(df, name="ALT", colorlist=["black", "lightsteelblue", "tomato"], figname=None):
    df1 = df.copy()
    fig, ax = plt.subplots(figsize=(7, 5))

    labellist = df1.index.tolist()
    df1.index = list(range(len(df1)))

    for i in range(len(df1)):
        x1 = [float(c.split("y")[1]) for c in df1.columns.tolist()]
        y1 = np.array(df1.loc[[i], :].values.tolist()[0])
        ax.plot(x1, y1, '^', color=colorlist[i], label=labellist[i], markersize=10)

    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set_xlabel("day", fontsize=20)
    ax.set_xticks(sorted(set(x1)))
    ax.tick_params(axis="both", labelsize=20)
    ax.set_ylabel(name, fontsize=20)
    ax.legend(fontsize=18)
    ax.grid(False)

    fig.tight_layout()
    if figname:
        save_panel(fig, figname, bbox_inches="tight")
    plt.show()

In [ ]:
plot_line(df_water, name="Amount of drink (g/individual)", figname=f"{fig_root}/fig_s1.png")

## = ad = ##

In [ ]:
def plot_patch_fraction_from_counts(source, output):
    from pathlib import Path
    from matplotlib.lines import Line2D
    scores = pd.read_csv(source)
    assert len(scores) == 35 and scores["slide_no"].is_unique
    scores["fraction"] = scores["abnormal_count"] / (scores["normal_count"] + scores["abnormal_count"])
    assert scores.query('group == "Control"')["training_overlap"].all()
    ref = scores.loc[scores.group == "Control", "fraction"].mean()
    fig, axes = plt.subplots(1, 3, figsize=(9, 5), sharey=True)
    palette = {"TAA": "coral", "MDA": "royalblue"}
    for ax, day in zip(axes, (7, 14, 28)):
        for x, group in enumerate(("TAA", "MDA")):
            v = scores.loc[(scores.group == group) & (scores.day == day), "fraction"].to_numpy(float)
            ax.bar(x, v.mean(), yerr=v.std(ddof=1), width=0.60, capsize=4,
                   color=palette[group], alpha=.85)
            ax.scatter(x + np.linspace(-.10, .10, len(v)), v,
                       c=palette[group], edgecolor="black", linewidth=.8, s=26)
        ax.axhline(ref, color="gray", linestyle="--")
        ax.set_title(f"Day {day}")
        ax.set_xticks((0, 1), ("TAA", "MDA"))
        ax.set_ylim(0, .18)
    axes[0].set_ylabel("Abnormal / total patches")
    fig.legend(handles=[Line2D((0,), (0,), color="gray", ls="--",
                  label="Control")],
               loc="lower center", frameon=False)
    fig.tight_layout(rect=(0, .07, 1, 1))
    save_panel(fig, output, bbox_inches="tight")
    plt.close(fig)

In [ ]:
plot_patch_fraction_from_counts(f"{data_root}/processed/fig2e_existing_patch_counts.csv", f"{fig_root}/fig_2e.png")